In [1]:
import pandas as pd
import re
from datetime import datetime

## Load OCR Dataset 

In [2]:
df = pd.read_csv("ocr_dataset.csv")

In [3]:
df.head()

,image,text,clean_text,dataset,is_total_line,price,date
0,E:\Projects_AI_ML\docfusion\data\raw\SROIE2019...,tan woon yann,tan woon yann,sroie,False,NaN,NaN
1,E:\Projects_AI_ML\docfusion\data\raw\SROIE2019...,BOOK TA.K (TAMAN DAYA) SDN BHD,book ta.k taman daya sdn bhd,sroie,False,NaN,NaN
2,E:\Projects_AI_ML\docfusion\data\raw\SROIE2019...,789417-W,789417w,sroie,False,NaN,NaN
3,E:\Projects_AI_ML\docfusion\data\raw\SROIE2019...,"NO.5: 55,57 & 59, JALAN SAGU 18,",no.5: 5557 59 jalan sagu 18,sroie,False,NaN,NaN
4,E:\Projects_AI_ML\docfusion\data\raw\SROIE2019...,"TAMAN DAYA,",taman daya,sroie,False,NaN,NaN


In [4]:
df.tail()

,image,text,clean_text,dataset,is_total_line,price,date
1162,E:\Projects_AI_ML\docfusion\data\raw\cord\trai...,Cash,cash,cord,False,NaN,NaN
1163,E:\Projects_AI_ML\docfusion\data\raw\cord\trai...,Tendered:,tendered:,cord,False,NaN,NaN
1164,E:\Projects_AI_ML\docfusion\data\raw\cord\trai...,50.000,50.000,cord,False,NaN,NaN
1165,E:\Projects_AI_ML\docfusion\data\raw\cord\trai...,Change:,change:,cord,False,NaN,NaN
1166,E:\Projects_AI_ML\docfusion\data\raw\cord\trai...,25.000,25.000,cord,False,NaN,NaN


In [6]:
df.columns

Index(['image', 'text', 'clean_text', 'dataset', 'is_total_line', 'price',
       'date'],
      dtype='object')

In [8]:
df.shape

(1167, 7)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1167 entries, 0 to 1166
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   image          1167 non-null   object 
 1   text           1162 non-null   object 
 2   clean_text     1148 non-null   object 
 3   dataset        1167 non-null   object 
 4   is_total_line  1167 non-null   bool   
 5   price          0 non-null      float64
 6   date           11 non-null     object 
dtypes: bool(1), float64(1), object(5)
memory usage: 56.0+ KB


### Create Receipt Groups

In [27]:
receipt_groups = df.groupby("image")
receipt_groups

## Vendor Extraction

In [135]:
def extract_vendor(receipt_df):

    top_lines = receipt_df.head(12)["clean_text"].dropna()

    ignore_keywords = [
        "tel", "fax", "receipt", "invoice",
        "document", "tax", "date", "time"
    ]

    company_keywords = [
        "sdn", "bhd", "enterprise", "trading",
        "shop", "mart", "store", "diy"
    ]

    candidates = []

    for line in top_lines:

        line = str(line)

        # skip tel/fax/system lines
        if any(word in line for word in ignore_keywords):
            continue

        # prefer company-like names
        if any(word in line for word in company_keywords):
            candidates.append(line)

    if candidates:
        return max(candidates, key=len)

    # fallback
    return max(top_lines, key=len)



# def extract_vendor(receipt_df):

#     top_lines = receipt_df.head(6)["clean_text"].dropna()

#     candidates = []

#     for line in top_lines:

#         line = str(line).strip()

#         # skip lines with many digits
#         digit_ratio = sum(c.isdigit() for c in line) / max(len(line),1)

#         if digit_ratio > 0.3:
#             continue

#         # skip very short lines
#         if len(line) < 5:
#             continue

#         candidates.append(line)

#     if candidates:
#         return max(candidates, key=len)

#     return top_lines.iloc[0]
  

## Date Extraction

In [136]:
import re

def extract_date(receipt_df):

    for text in receipt_df["clean_text"]:

        text = str(text)   # convert to string

        match = re.search(r'(\d{2}[/-]\d{2}[/-]\d{2,4})', text)

        if match:
            return match.group(1)

    return None

## Total Amount Extraction

In [137]:
import re

def extract_total(receipt_df):

    texts = receipt_df["clean_text"].astype(str)

    # Step 1: try lines containing total keywords
    total_lines = texts[texts.str.contains("total|amount", case=False, na=False)]

    for text in total_lines:
        match = re.search(r'(\d+\.\d{2})', text)
        if match:
            return match.group(1)

    # Step 2: fallback — take largest number in receipt
    numbers = []

    for text in texts:
        matches = re.findall(r'\d+\.\d{2}', text)
        numbers.extend(matches)

    if numbers:
        return max(numbers, key=float)

    return None

## Combine Extractors

In [138]:
def extract_receipt_fields(receipt_df):

    vendor = extract_vendor(receipt_df)

    date = extract_date(receipt_df)

    total = extract_total(receipt_df)

    return vendor, date, total

 ## Run Extraction on All Receipt

In [139]:
import os
records = []

for receipt_id, receipt_df in receipt_groups:

    vendor, date, total = extract_receipt_fields(receipt_df)

    records.append({
        "id": os.path.basename(receipt_id),
        "vendor": vendor,
        "date": date,
        "total": total
    })


In [147]:
structured_df = pd.DataFrame(records)

structured_df.head(5)

,id,vendor,date,total
0,X00016469612.jpg,book ta.k taman daya sdn bhd,25/12/2018,10.00
1,X00016469619.jpg,tel:073507405 fax:073558160,19/10/2018,70.30
2,X00016469620.jpg,mr d.i.y. johor sdn bhd,None,50.00
3,X00016469622.jpg,yongfatt enterprise,25/12/2018,100.00
4,X00016469623.jpg,mr d.i.y. m sdn bhd,None,51.00


In [92]:
structured_df.isnull().sum()

id         0
vendor     0
date      17
total      5
dtype: int64

In [141]:
from datetime import datetime

def normalize_date(date_str):
    if pd.isna(date_str):
        return None

    try:
        return datetime.strptime(date_str, "%d/%m/%Y").strftime("%Y-%m-%d")
    except:
        return date_str

In [142]:
structured_df["date"] = structured_df["date"].apply(normalize_date)

## Convert DataFrame to JSONL

In [145]:
import json

output_file = "predictions.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    for _, row in structured_df.iterrows():

        record = {
            "id": row["id"],
            "vendor": row["vendor"],
            "date": row["date"],
            "total": row["total"]
        }

        f.write(json.dumps(record) + "\n")

print("JSONL file created successfully!")

JSONL file created successfully!


In [146]:
with open("predictions.jsonl") as f:
    for i in range(5):
        print(f.readline())

{"id": "X00016469612.jpg", "vendor": "book ta.k taman daya sdn bhd", "date": "2018-12-25", "total": "10.00"}

{"id": "X00016469619.jpg", "vendor": "tel:073507405 fax:073558160", "date": "2018-10-19", "total": "70.30"}

{"id": "X00016469620.jpg", "vendor": "mr d.i.y. johor sdn bhd", "date": null, "total": "50.00"}

{"id": "X00016469622.jpg", "vendor": "yongfatt enterprise", "date": "2018-12-25", "total": "100.00"}

{"id": "X00016469623.jpg", "vendor": "mr d.i.y. m sdn bhd", "date": null, "total": "51.00"}

